# Chapter 10 — Prices and Inflation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch10_prices_and_inflation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 10.1 The CPI's groups and sub-groups

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from indiadata import mospi_cpi_groups

cpi = mospi_cpi_groups("combined")
print(cpi.shape, "| from", cpi["date"].min().strftime("%b %Y"), "to",
      cpi["date"].max().strftime("%b %Y"))
cpi.groupby("group")["subgroup"].nunique()

In [ ]:
all_months = pd.date_range("2013-01-01", "2025-12-01", freq="MS")
gaps = {}
for name, s in cpi.groupby("subgroup"):
    have = s.dropna(subset=["index"])["date"]
    missing = all_months.difference(have)
    if len(missing):
        gaps[name] = ", ".join(missing.strftime("%b %Y"))
pd.Series(gaps, name="months with no index value")

In [ ]:
cpi[cpi["status"] != "F"].groupby(["date", "status"]).size()

In [ ]:
overall = cpi[cpi["subgroup"].str.endswith("-Overall")]
idx = overall.pivot(index="date", columns="group", values="index")
published = overall.pivot(index="date", columns="group", values="inflation")

ours = 100 * idx.pct_change(12, fill_method=None)
gap = (ours - published).abs().max()
print("Largest gap, ours vs published inflation, by group (points):")
print(gap.round(3).to_string())

## 10.2 The official weights

In [ ]:
from pathlib import Path

import requests
from pypdf import PdfReader

PR_URL = ("https://mospi.gov.in/sites/default/files/press_release/"
          "cpi_pr_12feb15t.pdf")
pdf_path = Path("data/raw/cpi_pr_12feb15t.pdf")
if not pdf_path.exists():
    reply = requests.get(PR_URL, timeout=90, headers={
        "User-Agent": "python-for-indian-economics (teaching use)"})
    reply.raise_for_status()
    pdf_path.write_bytes(reply.content)

weights = {}
for line in PdfReader(pdf_path).pages[1].extract_text().splitlines():
    parts = line.split()
    is_group = bool(parts) and parts[0] in {"1", "2", "3", "4", "5", "6"}
    is_subgroup = parts and parts[0].count(".") == 2       # e.g. 1.1.07
    if len(parts) > 10 and (is_group or is_subgroup):
        weights[" ".join(parts[1:-9])] = float(parts[-3])  # combined weight
print(len(weights), "weights read")
print({k: weights[k] for k in ["Food and beverages", "Vegetables",
                               "Fuel and light", "Housing", "Miscellaneous"]})

In [ ]:
GROUPS = ["Food and Beverages", "Pan, Tobacco and Intoxicants",
          "Clothing and Footwear", "Housing", "Fuel and Light",
          "Miscellaneous"]
lower = {k.lower(): v for k, v in weights.items()}
w = pd.Series({g: lower[g.lower()] for g in GROUPS})
print(w.to_string())
print("Sum:", round(w.sum(), 2))

## 10.3 Contributions to headline inflation

In [ ]:
G = idx[GROUPS].copy()
G["Housing"] = G["Housing"].interpolate()      # fills the 10-month gap
rebuilt = G @ w / w.sum()

err = rebuilt - idx["General"]
in_gap = err.loc["2018-12":"2019-09"]
print(f"Rebuilt minus official, all months: largest |error| "
      f"{err.abs().max():.2f} index points")
print(f"Inside the interpolated Housing gap: largest |error| "
      f"{in_gap.abs().max():.2f}")

In [ ]:
contrib = (G.diff(12).mul(w, axis=1)
            .div(rebuilt.shift(12), axis=0) / w.sum() * 100).dropna()
headline = 100 * idx["General"].pct_change(12)

check = (contrib.sum(axis=1) - headline).abs()
print(f"Sum of contributions vs official headline inflation: largest gap "
      f"{check.max():.2f} points, average {check.mean():.3f}")
contrib.tail(3).round(2)

In [ ]:
SHORT = {"Food and Beverages": "Food & beverages",
         "Pan, Tobacco and Intoxicants": "Pan & tobacco",
         "Clothing and Footwear": "Clothing & footwear",
         "Housing": "Housing", "Fuel and Light": "Fuel & light",
         "Miscellaneous": "Miscellaneous (services etc.)"}
COLOURS = {"Food and Beverages": "#c55a11", "Pan, Tobacco and Intoxicants":
           "#e3a87a", "Clothing and Footwear": "#9db7d5", "Housing": "#7f7f7f",
           "Fuel and Light": "#3f7f3f", "Miscellaneous": "#1f4e79"}

fig, ax = plt.subplots(figsize=(10, 5))
pos = contrib.clip(lower=0)
neg = contrib.clip(upper=0)
colours = list(COLOURS.values())
ax.stackplot(contrib.index, pos.T.to_numpy(), colors=colours,
             labels=list(SHORT.values()), linewidth=0)
ax.stackplot(contrib.index, neg.T.to_numpy(), colors=colours, linewidth=0)
ax.plot(headline.loc[contrib.index[0]:], color="black", linewidth=1.4,
        label="Headline CPI inflation")
ax.axhline(4, color="black", linestyle="--", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.6)
ax.text(contrib.index[2], 4.15, "4% target", fontsize=8.5)
ax.set_ylabel("Percentage points (year on year)")
ax.set_title("What made up India's CPI inflation, month by month",
             loc="left", fontsize=13)
ax.legend(frameon=False, fontsize=8.5, ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.08))
ax.set_ylim(-2.5, 10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.text(0.01, 0.005, "CPI-Combined, base 2012. Housing interpolated Dec "
         "2018-Sep 2019. Source: MoSPI (eSankhyiki API; weights from press "
         "release of 12 Feb 2015); author's calculation.", fontsize=8,
         color="grey")
plt.show()

## 10.4 Is food driving inflation?

In [ ]:
fit = contrib.loc["2016-10":]
summary = pd.DataFrame({
    "weight, %": w,
    "share of average inflation, %": 100 * fit.mean() / fit.sum(axis=1).mean(),
}).rename(index=SHORT)
summary.round(2)

In [ ]:
total = fit.sum(axis=1)
var_share = 100 * fit.apply(lambda c: c.cov(total)) / total.var()
var_share.rename(index=SHORT).round(1)

In [ ]:
by_year = contrib.groupby(contrib.index.year).mean()
by_year = by_year.rename(columns=SHORT)
by_year["headline"] = by_year.sum(axis=1)
peak = by_year.loc[2017:, "headline"].idxmax()   # full inflation-targeting years
food_peak = by_year.loc[peak, "Food & beverages"]
print(f"Highest-inflation year since 2017: {peak}, "
      f"{by_year.loc[peak, 'headline']:.2f}%; "
      f"food contributed {food_peak:.2f} points "
      f"({100 * food_peak / by_year.loc[peak, 'headline']:.0f}%)")
by_year.loc[2017:, ["Food & beverages", "Clothing & footwear", "Fuel & light",
                  "Miscellaneous (services etc.)", "headline"]].rename(
    columns={"Miscellaneous (services etc.)": "Misc."}).round(2)

## 10.5 Core inflation: headline without food and fuel

In [ ]:
CORE = ["Pan, Tobacco and Intoxicants", "Clothing and Footwear", "Housing",
        "Miscellaneous"]
core_direct = idx[CORE] @ w[CORE] / w[CORE].sum()          # NaN in the gap
core_resid = ((idx["General"] * w.sum()
               - w["Food and Beverages"] * idx["Food and Beverages"]
               - w["Fuel and Light"] * idx["Fuel and Light"])
              / w[CORE].sum())

infl = pd.DataFrame({
    "headline": headline,
    "food": ours["Food and Beverages"],
    "fuel": ours["Fuel and Light"],
    "core": 100 * core_resid.pct_change(12),
    "core (direct)": 100 * core_direct.pct_change(12, fill_method=None),
}).loc["2014-01":]
diff = (infl["core"] - infl["core (direct)"]).abs()
print(f"Core inflation, residual vs direct: mean |gap| {diff.mean():.2f}, "
      f"largest {diff.max():.2f} points ({diff.notna().sum()} months)")
print(f"Weight of the core basket: {w[CORE].sum():.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axhspan(2, 6, color="tab:green", alpha=0.10, label="RBI band (2-6%)")
ax.plot(infl.index, infl["food"], color="#c55a11", linewidth=1.2,
        label="Food and beverages")
ax.plot(infl.index, infl["headline"], color="black", linewidth=1.6,
        label="Headline")
ax.plot(infl.index, infl["core"], color="#1f4e79", linewidth=2.2,
        label="Core (excl. food & fuel)")
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("Year-on-year, %")
ax.set_title("Headline, food and core CPI inflation", loc="left", fontsize=13)
ax.legend(frameon=False, ncol=4, fontsize=9, loc="upper center",
          bbox_to_anchor=(0.5, -0.07))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.text(0.01, 0.005, "Core = CPI excluding food & beverages and fuel & "
         "light, from group indices and 2012 weights. Source: MoSPI, CPI "
         "(base 2012); author's calculation.", fontsize=8, color="grey")
plt.show()

In [ ]:
since = infl.loc["2016-10":, ["headline", "food", "fuel", "core"]]
print("Averages since October 2016:")
print(since.mean().round(2).to_string())
both = since.dropna()
print(f"\nMonths with headline above 6%: {(both['headline'] > 6).sum()}, "
      f"of which core was also above 6%: "
      f"{((both['headline'] > 6) & (both['core'] > 6)).sum()}")
print(f"Months with headline below 2%: {(both['headline'] < 2).sum()}, "
      f"core in those months: {both.loc[both['headline'] < 2, 'core'].min():.2f}"
      f" to {both.loc[both['headline'] < 2, 'core'].max():.2f}%")
infl.loc["2025-06":, ["headline", "food", "fuel", "core"]].round(2)

## 10.6 Volatility: rolling windows

In [ ]:
infl[["headline", "food", "fuel", "core"]].loc["2014-01":].std().round(2)

In [ ]:
vol = infl[["headline", "food", "core"]].rolling(24).std()
print(f"Highest food volatility: {vol['food'].max():.2f} points, 24 months "
      f"to {vol['food'].idxmax():%b %Y}")
vol.dropna().iloc[::24].round(2)            # every 24th month, to save space

## 10.7 Seasonality: why vegetable prices rise every autumn

In [ ]:
veg = cpi[cpi["subgroup"] == "Vegetables"].set_index("date")["index"]
veg = veg.asfreq("MS")          # tell pandas the data are monthly
print(veg.isna().sum(), "missing months;", len(veg), "months")

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

parts = seasonal_decompose(veg, model="multiplicative", period=12)
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep",
          "Oct", "Nov", "Dec"]
seas_veg = 100 * (parts.seasonal.groupby(parts.seasonal.index.month).mean() - 1)
seas_veg.index = MONTHS

general = idx["General"].asfreq("MS")
seas_gen = 100 * (seasonal_decompose(general, model="multiplicative",
                                     period=12).seasonal
                  .groupby(general.index.month).mean() - 1)
seas_gen.index = MONTHS
pd.DataFrame({"vegetables, % vs trend": seas_veg,
              "general CPI, % vs trend": seas_gen}).round(1).T

## 10.8 Persistence: do food shocks last?

In [ ]:
LAGS = [1, 3, 6, 12]
persist = pd.DataFrame({
    col: [infl[col].loc["2014-01":].autocorr(k) for k in LAGS]
    for col in ["headline", "food", "fuel", "core"]},
    index=[f"{k} months" for k in LAGS])
persist.round(2)

## 10.9 CPI versus WPI

In [ ]:
from indiadata import mospi_wpi

pieces = []
for year in range(2013, 2026):
    one = mospi_wpi(years=[year])
    pieces.append(one.loc[one["group"].isna(), ["date", "majorgroup", "index"]])
    del one                                   # free the big table at once
wpi = (pd.concat(pieces).drop_duplicates(["date", "majorgroup"])
         .pivot(index="date", columns="majorgroup", values="index"))
del pieces
print(wpi.shape)
wpi.tail(3)

In [ ]:
cmp_ = pd.DataFrame({
    "CPI": infl["headline"],
    "WPI": 100 * wpi["Wholesale price index"].pct_change(12),
    "WPI manufactured": 100 * wpi["Manufactured products"].pct_change(12),
    "WPI fuel & power": 100 * wpi["Fuel & power"].pct_change(12),
}).dropna()
cmp_["CPI minus WPI"] = cmp_["CPI"] - cmp_["WPI"]
print(f"{cmp_.index[0]:%b %Y} to {cmp_.index[-1]:%b %Y}: {len(cmp_)} months")
print(f"Correlation of CPI and WPI inflation: "
      f"{cmp_['CPI'].corr(cmp_['WPI']):.2f}")
print(f"CPI above WPI in {(cmp_['CPI minus WPI'] > 0).mean():.0%} of months")
print(f"Largest gap: {cmp_['CPI minus WPI'].abs().idxmax():%b %Y} "
      f"({cmp_.loc[cmp_['CPI minus WPI'].abs().idxmax(), 'CPI minus WPI']:+.1f})")
cmp_.groupby(cmp_.index.year).mean().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(cmp_.index, cmp_["CPI"], color="#1f4e79", linewidth=2, label="CPI")
ax.plot(cmp_.index, cmp_["WPI"], color="#c55a11", linewidth=2, label="WPI")
ax.fill_between(cmp_.index, cmp_["CPI"], cmp_["WPI"], color="#9db7d5",
                alpha=0.35, label="Gap")
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("Year-on-year, %")
ax.set_title("Retail and wholesale inflation: CPI vs WPI", loc="left",
             fontsize=13)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "CPI-Combined (base 2012) and WPI all commodities (base "
         "2011-12). Source: MoSPI and Office of the Economic Adviser, via "
         "eSankhyiki API.", fontsize=8, color="grey")
plt.show()

## 10.10 Answering the question

## Key ideas

## Exercises

## Data citation

## Further reading